# 11 · Streaming responses
Level L2 · Part 2 · core · ~30 min · builds on 07

## Goal
Stream a reply from Gemini, print it piece by piece as it arrives, and measure **time to first token**
(TTFT), total time and **tokens per second**, then compare them with a normal (blocking) call.

## The idea
In lesson 07, `generate_content` waited until the whole reply was written, then handed it over in one
piece. A long reply can take several seconds, and the user stares at nothing. **Streaming** asks the
server to send the reply in small **chunks** while the model is still writing. The total time stays
about the same. What changes is when the first words appear: the time to first token.

Analogy: a blocking call is a letter that arrives when the writer has finished. A stream is a phone
call: you hear the first words almost at once, and the conversation takes just as long.

## Picture
```mermaid
sequenceDiagram
    participant S as Scout (your code)
    participant G as Gemini
    S->>G: POST ...:streamGenerateContent?alt=sse
    G-->>S: data: {chunk 1 text}   (time to first token)
    G-->>S: data: {chunk 2 text}
    G-->>S: data: {...}
    G-->>S: data: {last chunk: finish_reason + usage}
    Note over S: total time = arrival of the last chunk
```

## Step by step
### 1. Start the recorder
As in lesson 07, every model call is recorded once and replayed from
`../data/llm_cassettes/11_streaming.yaml`, so this notebook runs with no key.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("11_streaming")

LLM mode: replay (11_streaming.yaml)


### 2. Setup from lesson 07
Scout's notes, the pinned model id, one client, and a config with automatic function calling switched off.

In [2]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
CONFIG = types.GenerateContentConfig(
    max_output_tokens=400,
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
print(MODEL, "|", len(NOTES), "notes")

gemini-3.5-flash-lite | 6 notes


### 3. The smallest stream: a fake model
Before the real model, a fake one. A Python **generator** that sleeps 0.5 s ("thinking about the
prompt"), then yields one word every 0.05 s, behaves just like a stream. The blocking version
collects every word first and returns the whole text.

In [3]:
import time

ANSWER = "Mars has two small moons and a day of about 24 hours and 39 minutes."

def fake_stream(text, first_delay=0.5, per_word=0.05):
    time.sleep(first_delay)
    for word in text.split(" "):
        yield word + " "
        time.sleep(per_word)

def fake_blocking(text):
    return "".join(fake_stream(text))

print(fake_blocking(ANSWER))

Mars has two small moons and a day of about 24 hours and 39 minutes. 


### 4. Measure both ways
`time.perf_counter()` is a precise stopwatch. For the blocking call we can only time the end.
For the stream we note when the first chunk arrives (TTFT) and when the last one does (total).
`print(..., end="", flush=True)` shows each chunk as soon as it arrives, on one line.

In [4]:
start = time.perf_counter()
fake_blocking(ANSWER)
print(f"blocking : first words after {time.perf_counter() - start:.1f} s")

start, first = time.perf_counter(), None
for piece in fake_stream(ANSWER):
    first = first or time.perf_counter() - start
    print(piece, end="", flush=True)
total = time.perf_counter() - start
print(f"\nstreaming: first words after {first:.1f} s, done after {total:.1f} s")

blocking : first words after 1.3 s


Mars 

has 

two 

small 

...

hours 

and 

39 

minutes. 


streaming: first words after 0.5 s, done after 1.3 s


Both take about 1.3 s in total. The stream showed its first word after 0.5 s, the blocking call after
1.3 s. Streaming does not make the model faster; it lets the user start reading sooner.

### 5. On the wire: server-sent events
How does a stream travel over HTTP? Gemini uses **server-sent events (SSE)**: one HTTP response that
stays open, with lines starting `data: ` and a blank line after each event. Let's skip the SDK and
ask for one with `httpx` (the key travels in a header, which the recorder strips).

In [5]:
import httpx

url = f"https://aiplatform.googleapis.com/v1beta1/publishers/google/models/{MODEL}:streamGenerateContent"
body = {"contents": [{"role": "user", "parts": [{"text": "List the two moons of Mars."}]}],
        "generationConfig": {"maxOutputTokens": 30}}
raw_lines = []
with httpx.stream("POST", url, params={"alt": "sse"}, json=body, timeout=60,
                  headers={"x-goog-api-key": os.environ.get("GOOGLE_CLOUD_API_KEY", "")}) as r:
    print(r.status_code, r.headers["content-type"])
    raw_lines = list(r.iter_lines())
for line in raw_lines:
    print(repr(line[:90]))

200 text/event-stream
'data: {"candidates": [{"content": {"role": "model","parts": [{"text": "The two"}]}}],"usag'
''
'data: {"candidates": [{"content": {"role": "model","parts": [{"text": " moons of Mars are:'
''
'data: {"candidates": [{"content": {"role": "model","parts": [{"text": "**"}]}}],"usageMeta'
''
'data: {"candidates": [{"content": {"role": "model","parts": [{"text": "","thoughtSignature'
''


Content type `text/event-stream`, then events. Each `data:` line is a complete JSON object, the same
shape as lesson 07's response, holding only the new piece of text. Let's decode them.

In [6]:
import json

events = [json.loads(line.removeprefix("data: ")) for line in raw_lines if line.startswith("data: ")]
for i, ev in enumerate(events):
    cand = ev["candidates"][0]
    text = "".join(p.get("text", "") for p in cand["content"]["parts"])
    print(i, repr(text), "| finish:", cand.get("finishReason"), "| usage:", ev["usageMetadata"])

0 'The two' | finish: None | usage: {'trafficType': 'ON_DEMAND'}
1 ' moons of Mars are:\n\n1. **Phobos**\n2. **Deimos' | finish: None | usage: {'trafficType': 'ON_DEMAND'}
2 '**' | finish: None | usage: {'trafficType': 'ON_DEMAND'}
3 '' | finish: STOP | usage: {'promptTokenCount': 7, 'candidatesTokenCount': 21, 'totalTokenCount': 28, 'trafficType': 'ON_DEMAND', 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 7}], 'candidatesTokensDetails': [{'modality': 'TEXT', 'tokenCount': 21}]}


Only the last event carries `finishReason`, and only it has the token counts in `usageMetadata`. Keep
that in mind; it matters for cost in lesson 12.

### 6. Scout streams a long answer with the SDK
The SDK does the SSE parsing for you: `client.models.generate_content_stream` returns an iterator of
response chunks. Scout gets the three Mars notes and is asked for a longer explanation, so there is
something to watch. We record when each chunk arrives.

In [7]:
mars = "\n".join(f"Note {n['id']}: {n['text']}" for n in NOTES if "Mars" in n["text"])
question = f"{mars}\n\nUsing only these notes, write about 150 words explaining Mars to a 12-year-old."

chunks, times = [], []
start = time.perf_counter()
stream = client.models.generate_content_stream(model=MODEL, contents=question, config=CONFIG)
for chunk in stream:
    times.append(time.perf_counter() - start)
    chunks.append(chunk)
    print(chunk.text or "", end="", flush=True)
print(f"\n\n{len(chunks)} chunks")

Hey there

! Did you know that Mars is a really cool planet with some amazing facts? First

 of all, a day on Mars is super similar to a day here on Earth. In fact, a Martian day lasts about

 24 hours and 39 minutes, so you wouldn't even have to change your watch very much! 

If you

 looked up into the night sky from Mars, you would see two tiny moons instead of just one like we have on Earth. These two small

 moons are named Phobos and Deimos. 

Space scientists are very interested in exploring Mars. They even sent a special robot called the

 rover Perseverance all the way there, and it successfully landed on the planet in February 2021 to check things

 out. Isn't space exploration awesome?



9 chunks


### 7. What is in each chunk
Each chunk is a full response object (lesson 07) holding a slice of the text. Let's print the size of
each slice, its finish reason and its output-token count.

In [8]:
for i, ch in enumerate(chunks):
    reason = ch.candidates[0].finish_reason
    out = ch.usage_metadata.candidates_token_count if ch.usage_metadata else None
    print(f"{i:2}  {len(ch.text or ''):3} chars  finish={reason.value if reason else None}  output_tokens={out}")

 0    9 chars  finish=None  output_tokens=None
 1   79 chars  finish=None  output_tokens=None
 2   98 chars  finish=None  output_tokens=None
 3   92 chars  finish=None  output_tokens=None
 4  128 chars  finish=None  output_tokens=None
 5  135 chars  finish=None  output_tokens=None
 6  112 chars  finish=None  output_tokens=None
 7   38 chars  finish=None  output_tokens=None
 8    0 chars  finish=STOP  output_tokens=161


Same as on the wire: the text comes in pieces, and only the last chunk says `STOP` and gives the
token count. To get the whole reply, join the pieces.

In [9]:
full_text = "".join(ch.text or "" for ch in chunks)
last = chunks[-1]
out_tokens = last.usage_metadata.candidates_token_count + (last.usage_metadata.thoughts_token_count or 0)
print(len(full_text), "characters,", out_tokens, "output tokens, finish:", last.candidates[0].finish_reason.value)

691 characters, 161 output tokens, finish: STOP


### 8. The same question, blocking
For a fair comparison, ask the same question with the normal `generate_content` and time it.
Here the first word and the last word reach you at the same moment.

In [10]:
start = time.perf_counter()
blocking = client.models.generate_content(model=MODEL, contents=question, config=CONFIG)
blocking_total = time.perf_counter() - start
print(len(blocking.text), "characters, finish:", blocking.candidates[0].finish_reason.value)

734 characters, finish: STOP


### 9. Real timings, kept with the recording
The recorder reads each response to the end before handing it over, and a replay hands back every
chunk instantly. So timings measured while recording or replaying mean nothing. The real timings come
from a separate live run (`TUT_LLM_MODE=live TUT_SAVE_TIMINGS=1`) saved to `../data/recorded/11.json`;
we load that file and label it as recorded output. With your own key in `live` mode you see your own numbers.

In [11]:
from pathlib import Path

TIMINGS = Path("../data/recorded/11.json")
measured = {"stream_times": [round(t, 3) for t in times], "blocking_total": round(blocking_total, 3),
            "chunk_texts": [ch.text or "" for ch in chunks], "output_tokens": out_tokens}
if llm_replay.mode() == "live":
    timing = measured
    if os.environ.get("TUT_SAVE_TIMINGS"):
        TIMINGS.write_text(json.dumps(measured, indent=1))
else:
    print(f"{llm_replay.mode()} mode: first chunk {times[0]:.2f} s, last {times[-1]:.2f} s (all at once).")
    print(f"Using recorded output from a live run: {TIMINGS.name}")
    timing = json.loads(TIMINGS.read_text())
print(len(timing["stream_times"]), "chunk times, blocking total", timing["blocking_total"], "s")

replay mode: first chunk 0.06 s, last 0.07 s (all at once).
Using recorded output from a live run: 11.json
9 chunk times, blocking total 1.156 s


### 10. TTFT, total time and tokens per second
**TTFT** is when the first chunk arrived. **Total** is when the last one did. **Tokens per second** is
how fast the model writes once it has started: output tokens divided by the time from first to last chunk.

In [12]:
def stream_stats(stream_times, output_tokens):
    ttft, total = stream_times[0], stream_times[-1]
    return {"ttft_s": round(ttft, 2), "total_s": round(total, 2),
            "tokens_per_s": round(output_tokens / (total - ttft))}

stats = stream_stats(timing["stream_times"], timing["output_tokens"])
print("streaming:", stats)
print("blocking : first words after", round(timing["blocking_total"], 2), "s")

streaming: {'ttft_s': 0.53, 'total_s': 1.23, 'tokens_per_s': 234}
blocking : first words after 1.16 s


Read the numbers above. The stream showed its first words after about half a second; the blocking
call showed nothing until it had finished, after more than a second. Both finished at roughly the same
time (they are separate calls, so totals vary a little). On longer replies the gap grows. Tokens per second is a property of the model and server load, and it is the number to compare
when you pick a model (lesson 17).

### 11. Watch it at the real pace
To feel the difference, replay the recorded chunks with their recorded delays.

In [13]:
def replay_paced(texts, stream_times):
    start = time.perf_counter()
    for text, at in zip(texts, stream_times):
        time.sleep(max(0.0, at - (time.perf_counter() - start)))
        print(text, end="", flush=True)
    print(f"\n[{time.perf_counter() - start:.1f} s]")

replay_paced(timing["chunk_texts"], timing["stream_times"])

Mars is

 an amazing planet with some really cool features! Did you know that a day on Mars is

 very similar to a day on Earth? In fact, a Martian day lasts about 24 hours and 39 minutes,

 so the daily schedule there would feel almost the same as ours. 

If you were to look up into the Martian night sky, you would

 see two tiny moons instead of just one like Earth has. These two small moons are named Phobos and Deimos.

 

Scientists are super interested in Mars, and we are learning more about it all the time. Humans have even sent robotic

 explorers there to check it out. For example, the rover named Perseverance successfully landed on Mars in February 2021 to explore

 its surface and help us learn more about our planetary neighbor!


[1.2 s]


## At work
- **Chat products stream every reply, and track TTFT.** What users feel is the wait for the first
  words, so teams log TTFT, total time and tokens/sec per call and per model, next to the token usage.
  → lesson 49
- **Usage and the stop reason come only in the last chunk.** Billing code that reads them early records
  nothing, and a stream cut off by a network error has no usage at all. → lessons 12 and 13
- **"Streaming makes it faster" is a common misconception.** Total time is the same; only the
  perceived wait changes. Measure both before promising latency numbers. → lesson 17
- **Agents stream too, including tool calls**, whose arguments arrive as partial JSON that you must
  collect before parsing. → lesson 36

## What just happened
- A fake generator showed the idea: same total time, but the first word appears after 0.5 s instead of 1.3 s.
- A raw `httpx` request showed server-sent events: `data: {json}` lines, with `finishReason` and token counts only in the last one.
- `generate_content_stream` gave Scout's answer as chunks we printed as they arrived.
- Joining the chunks gave the full text; the last chunk gave `STOP` and the output tokens.
- Real timings from recording time gave TTFT, total time and tokens/sec, and the blocking call's first words came at its total time.

## Common mistakes
**1. Reading usage from the first chunk.** Early chunks have no token counts, so the maths fails.

In [14]:
first_chunk = chunks[0]
print(first_chunk.usage_metadata.candidates_token_count / stats["total_s"], "tokens/s")

TypeError: unsupported operand type(s) for /: 'NoneType' and 'float'

Fix: take usage and the finish reason from the **last** chunk, after the stream has ended.

In [15]:
print(round(chunks[-1].usage_metadata.candidates_token_count / stats["total_s"]), "output tokens per second of total time")

131 output tokens per second of total time


**2. Reading a stream twice.** A stream is an iterator: once you have looped over it, it is empty.
Asking for one more chunk raises `StopIteration`, and a second `for` loop silently prints nothing.

In [16]:
next(stream)

StopIteration: 

Fix: keep the chunks in a list as they arrive (as we did with `chunks`) and reuse the list.

In [17]:
print("".join(ch.text or "" for ch in chunks)[:80], "...")

Hey there! Did you know that Mars is a really cool planet with some amazing fact ...


## Try it
A voice assistant can start speaking once a full sentence has arrived. Using `timing["chunk_texts"]`
and `timing["stream_times"]`, write `first_sentence_at(texts, stream_times)` that returns the time
when the joined text first contains a `.`, `!` or `?`. Compare it with the TTFT and the blocking total.
No new model calls needed.

In [18]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [19]:
llm_replay.stop()

## Key terms
- **streaming** — receiving a model's reply in small pieces while it is still being generated, instead of all at once at the end.
- **chunk** — one piece of a streamed reply; in Gemini a full response object holding only the new text.
- **server-sent events (SSE)** — an HTTP response that stays open and sends `data: ...` lines, one event per blank-line-separated block; content type `text/event-stream`.
- **time to first token (TTFT)** — the time from sending the request to receiving the first piece of the reply; what users feel as waiting.
- **tokens per second** — how fast the model writes once it has started: output tokens divided by the time from first to last chunk.